In [1]:
import os
os.chdir('../')

In [2]:
import torch
from config.gpt_config import GPTConfig
from model.llm import LLM
from training.trainer import load_checkpoint
from tokenizer.bpe import BPETokenizer

In [3]:
device = torch.device("cpu")

tokenizer = BPETokenizer()
tokenizer.load("data/vocab.json", "data/merges.json")

model = LLM(GPTConfig()).to(device)
optimizer = torch.optim.AdamW(model.parameters())

load_checkpoint("checkpoints/ckpt_epoch3.pt", model, optimizer, device)
model.eval()

Resumed from checkpoints/ckpt_epoch3.pt (epoch 3, step 20742, mid_epoch=False)


LLM(
  (tok_emb): Embedding(8192, 768)
  (pos_emb): Embedding(512, 768)
  (drop_emb): Dropout(p=0.1, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (attn): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=False)
        (W_key): Linear(in_features=768, out_features=768, bias=False)
        (W_value): Linear(in_features=768, out_features=768, bias=False)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.1, inplace=False)
    )
    (1): TransformerBlock(
      (attn): MultiHeadAttention(
        (W_query): Linear(in_features=768

In [4]:
from model.generate import generate_text_simple, text_to_token_idx, token_idx_to_text

In [8]:
def generate(model, idx, max_new_tokens, context_size, temperature=1.0, top_k=None):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)

        logits = logits[:, -1, :] / temperature

        if top_k is not None:
            top_values, _ = torch.topk(logits, top_k)
            logits[logits < top_values[:, -1:]] = float("-inf")

        probs = torch.softmax(logits, dim=-1)
        idx_next = torch.multinomial(probs, num_samples=1)
        idx = torch.cat((idx, idx_next), dim=1)
    return idx



start_context = "There was"
token_ids = generate(
    model=model,
    idx=text_to_token_idx(start_context, tokenizer),
    max_new_tokens=200,
    context_size=GPTConfig().context_length,
    temperature=0.8,
    top_k=40,
)
print(token_idx_to_text(token_ids, tokenizer))

There was a little girl named Mia. Mia liked to play in the park. She liked to run, jump, and laugh. One day, Mia saw a big tree with a swing. The swing was very high. Mia wanted to play on the swing, but she was too small.

Mia saw a boy on the swing. He was trying to push her. But the swing was broken. Mia said, "No, you can't push me." The boy was sad.

Mia had an idea. She went to her house and called her mom. Her mom came and fixed the swing. Mia was happy and had lots of fun on the swing. The end.
One day, a little girl named Mia went to the park with her mom. Mia saw a boy with a big toy car. She wanted to play with the car too.

"Hi, can I play with your car?" Mia asked the boy. The boy looked at her and said, "No, this is mine. You can't play with it."

Mia felt sad and went to sit on a bench. A kind boy named Tim saw her and came over. He said, "I like


In [6]:
import math
from datasets import load_dataset
from data.dataset import create_dataloader_from_ids
from training.trainer import calc_loss_loader

ds_val = load_dataset("roneneldan/TinyStories", split="validation[:500]")
val_text = "\n".join(ds_val["text"])
val_ids = tokenizer.encode(val_text)

val_loader = create_dataloader_from_ids(
    val_ids,
    max_length=GPTConfig().context_length,
    stride=GPTConfig().context_length,
    batch_size=8,
    shuffle=False,
    drop_last=False,
    num_workers=0,
)

with torch.no_grad():
    val_loss = calc_loss_loader(val_loader, model, device, num_batches=50)

perplexity = math.exp(val_loss)
print(f"Val loss:    {val_loss:.4f}")
print(f"Perplexity:  {perplexity:.2f}")

d:\GptFromScratch\gpt_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Val loss:    1.3231
Perplexity:  3.75
